# Analyse et optimisation des performances du modèle

Ce notebook mesure les performances actuelles de l'API de crédit scoring, identifie les goulots d'étranglement via profiling (`cProfile`), teste des optimisations, puis compare les résultats avant/après.

## Profiling initial

On utilise `cProfile` pour mesurer, fonction par fonction, où le temps est réellement dépensé lors d'un appel à `predict()`. On répète l'appel 50 fois pour lisser les micro-variations de mesure et obtenir un temps cumulé significatif.

In [2]:
import cProfile
import json
import os
import pstats
import sys

sys.path.append("..")                                             # permet d'importer app.* depuis le dossier notebooks
os.chdir("..")                                                    # place le dossier de travail à la racine du projet

from app.model import load_model, predict

model = load_model()                                               # chargement du modele, comme au démarrage de l'API
exemple_client = {"AMT_INCOME_TOTAL": 202500, "EXT_SOURCE_2": 0.262}  # un client type, réutilisé à chaque appel

def repeter_predictions(n=50):
    for _ in range(n):
        predict(model, exemple_client)                             # simule 50 appels successifs à predict()

profiler = cProfile.Profile()                                      # chronometre chaque fonction appelée
profiler.enable()                                                  # démarre le chronometrage
repeter_predictions(50)                                            # code à mesurer : 50 predictions
profiler.disable()                                                 # arrete le chronometrage

stats = pstats.Stats(profiler)
stats.sort_stats("cumulative")                                       # trie les fonctions par temps cumulé (les plus longues en premier)
stats.print_stats(15)                                                 # le top 15 + lisible

         2455882 function calls (2365406 primitive calls) in 1.351 seconds

   Ordered by: cumulative time
   List reduced from 323 to 15 due to restriction <15>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        2    0.000    0.000    1.351    0.675 c:\Users\yavas\credit-scoring-mlops\venv\Lib\site-packages\IPython\core\interactiveshell.py:3786(run_code)
        2    0.000    0.000    1.351    0.675 {built-in method builtins.exec}
        1    0.000    0.000    1.343    1.343 C:\Users\yavas\AppData\Local\Temp\ipykernel_13624\3603416480.py:1(<module>)
        1    0.002    0.002    1.343    1.343 C:\Users\yavas\AppData\Local\Temp\ipykernel_13624\3603416480.py:15(repeter_predictions)
       50    0.016    0.000    1.341    0.027 c:\Users\yavas\credit-scoring-mlops\notebooks\..\app\model.py:19(predict)
       50    0.001    0.000    1.023    0.020 c:\Users\yavas\credit-scoring-mlops\venv\Lib\site-packages\pandas\core\frame.py:698(__init__)
       50    0.001

### Interprétation profiling

Le profiling sur 50 appels à `predict()` révèle un temps total de 1.341 s, soit environ **27 ms par prédiction** (cohérent avec les latences observées dans les logs de production).

**Répartition du temps** :
- `predict()` elle-même ne passe que 16 ms de temps propre (`tottime`) sur les 1.341 s cumulées. L'essentiel du temps est donc dépensé dans les fonctions qu'elle appelle, pas dans sa propre logique.
- La construction du DataFrame pandas (`sanitize_array`) apparaît comme un poste de dépense notable : plus de 42 000 appels à des fonctions de construction pandas sont recensés pour seulement 50 prédictions, soit environ 843 appels par prédiction (un chiffre proche du nombre de features (839)), ce qui suggère que pandas traite les colonnes individuellement.

**Goulot d'étranglement identifié** : la construction du DataFrame pandas. En effet, pandas doit inspecter et traiter chaque clé du dictionnaire individuellement pour construire les colonnes, ce qui explique les 843 appels internes par prédiction.


## Optimisation : remplacer le DataFrame par un array NumPy

On teste une version alternative de `predict()` qui construit un tableau NumPy plutôt qu'un DataFrame pandas, pour réduire le coût de construction identifié par le profiling. Avant de mesurer le gain de vitesse, on vérifie que le résultat de prédiction reste identique (**test de non-régression**). J'intègre à la version NumPy un chargement unique des feature_names au démarrage au lieu d'etre relu à chaque requete (double optimisation?).  
### Test de non régression

In [ ]:
# test de non régression 
import numpy as np
import json

def predict_numpy(model, feature_names, features):
    """Version optimisee de predict(), utilisant un array NumPy plutot qu'un DataFrame."""
    valeurs = [features.get(name, np.nan) for name in feature_names]
    X = np.array([valeurs])
    proba = model.predict_proba(X)
    return float(proba[0, 1])

# Lecture directe du fichier json pour feature_names
with open("models/feature_names.json") as f:
    feature_names = json.load(f)

# Comparaison sur le meme client (la proba de défaut est-elle la meme?)
resultat_original = predict(model, exemple_client)                          # signature actuelle de app/model.py (2 arguments)
resultat_numpy = predict_numpy(model, feature_names, exemple_client)        # version testée (3 arguments pour charger le modèle et la liste des features une seule fois au démarrage)

print("Version originale (DataFrame) :", resultat_original)
print("Version optimisee (NumPy)     :", resultat_numpy)
print("Resultats identiques :", resultat_original == resultat_numpy)

Version originale (DataFrame) : 0.6961562939778133
Version optimisee (NumPy)     : 0.6961562939778133
Resultats identiques : True


c:\Users\yavas\credit-scoring-mlops\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


### Comparaison des performances : DataFrame vs NumPy

Les deux versions produisent des résultats identiques. On compare maintenant leur temps d'exécution sur 50 appels, dans les mêmes conditions que le profiling initial.

In [6]:
import time

# Version originale (DataFrame)
debut = time.perf_counter()
for _ in range(50):
    predict(model, exemple_client)
temps_original = time.perf_counter() - debut

# Version optimisée (NumPy)
debut = time.perf_counter()
for _ in range(50):
    predict_numpy(model, feature_names, exemple_client)
temps_optimisee = time.perf_counter() - debut

print(f"Version originale (DataFrame) : {temps_original*1000:.1f} ms pour 50 appels ({temps_original/50*1000:.2f} ms/appel)")
print(f"Version optimisée (NumPy)     : {temps_optimisee*1000:.1f} ms pour 50 appels ({temps_optimisee/50*1000:.2f} ms/appel)")
print(f"Gain : {(1 - temps_optimisee/temps_original)*100:.1f}%")

Version originale (DataFrame) : 555.0 ms pour 50 appels (11.10 ms/appel)
Version optimisée (NumPy)     : 85.7 ms pour 50 appels (1.71 ms/appel)
Gain : 84.6%


c:\Users\yavas\credit-scoring-mlops\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\yavas\credit-scoring-mlops\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\yavas\credit-scoring-mlops\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\yavas\credit-scoring-mlops\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\yavas\credit-scoring-mlops\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier wa

### Conclusion

Le remplacement de la construction du DataFrame pandas par un tableau NumPy réduit le temps d'inférence de **84.6%** (11.10 ms → 1.71 ms par prédiction), sans aucune régression sur le résultat.

**Décision** : on intègre cette optimisation dans `app/model.py`.